# Quick Training Demo on GIST Data

This notebook shows a **minimal training example** on a few GIST cases.

⚠️ **This is a DEMO only** - for real training use the full pipeline!

In [ ]:
import sys
sys.path.insert(0, r'C:\Users\cahel\Desktop\Med3Tab-PFN')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import yaml
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm import tqdm

from geotopo_sts.utils import set_random_seed
set_random_seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1. Load Config

In [ ]:
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Quick demo settings - reduce for speed
config['geometry']['mesh']['target_vertices'] = 3000  # Faster
config['preprocessing']['crop_size'] = [128, 128, 128]  # Smaller

print("Config loaded and adjusted for quick demo")

## 2. Load a Few GIST Cases

In [ ]:
from geotopo_sts.gist_data_loader import discover_gist_cases, load_gist_case
from geotopo_sts.dataio.preprocess import preprocess_case
from geotopo_sts.geometry import extract_mesh_from_mask, compute_mesh_node_features, build_mesh_graph
from geotopo_sts.topology import extract_ph_features

gist_root = r'C:\Users\cahel\Desktop\Med3Tab-PFN\data\gist'
cases = discover_gist_cases(gist_root)

# Use first 5 cases for quick demo
demo_cases = cases[:5]
print(f"Using {len(demo_cases)} cases for demo")

## 3. Preprocess Cases

In [ ]:
def process_case(case_info, config):
    """Process one GIST case"""
    # Load
    volume, mask, spacing = load_gist_case(case_info)
    
    # Preprocess
    preprocessed = preprocess_case(volume, mask, spacing, 'ct', config['preprocessing'])
    
    # Extract mesh
    vertices, faces = extract_mesh_from_mask(
        preprocessed['mask'],
        spacing=preprocessed['spacing'],
        target_vertices=config['geometry']['mesh']['target_vertices']
    )
    
    if len(vertices) > 0:
        node_features = compute_mesh_node_features(
            vertices, faces,
            preprocessed['volume'],
            preprocessed['mask'],
            preprocessed['rim'],
            spacing=preprocessed['spacing']
        )
        edge_index, edge_attr = build_mesh_graph(vertices, faces, k_neighbors=8)
    else:
        node_features = np.zeros((100, 9))  # Fallback
        vertices = np.zeros((100, 3))
        edge_index = np.array([[i, (i+1)%100] for i in range(100)]).T
    
    # Topology features (skip if too slow)
    try:
        ph_features = extract_ph_features(
            preprocessed['mask'],
            preprocessed['rim'],
            preprocessed['volume'],
            spacing=preprocessed['spacing'],
            config=config['topology']
        )
    except:
        ph_features = np.zeros(128)
    
    return {
        'volume': preprocessed['volume'],
        'mask': preprocessed['mask'],
        'rim': preprocessed['rim'],
        'vertices': vertices,
        'node_features': node_features,
        'edge_index': edge_index,
        'ph_features': ph_features,
        'case_id': case_info['case_id']
    }

# Process all demo cases
print("Processing cases...")
processed_data = []
for case_info in tqdm(demo_cases):
    try:
        data = process_case(case_info, config)
        processed_data.append(data)
    except Exception as e:
        print(f"Error processing {case_info['case_id']}: {e}")

print(f"Successfully processed {len(processed_data)} cases")

## 4. Create Model

In [ ]:
from geotopo_sts.models import GeoTopoSTS

# Create model
model = GeoTopoSTS(config['model'])
model.train()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"Model created on {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

## 5. Quick Training Loop (Demo)

In [ ]:
# Create optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

# Demo labels (random for this example - replace with real labels!)
np.random.seed(42)
labels = np.random.randint(0, config['model']['head']['n_classes'], len(processed_data))

# Training loop
n_epochs = 10
losses = []

print(f"\nTraining for {n_epochs} epochs...")
for epoch in range(n_epochs):
    epoch_loss = 0
    
    for i, data in enumerate(processed_data):
        # Prepare batch
        batch = {
            'volume': torch.from_numpy(data['volume'][None, None, ...]).float().to(device),
            'mask': torch.from_numpy(data['mask'][None, None, ...]).float().to(device),
            'rim': torch.from_numpy(data['rim'][None, None, ...]).float().to(device),
            'mesh_data': {
                'vertices': torch.from_numpy(data['vertices']).float().to(device),
                'features': torch.from_numpy(data['node_features']).float().to(device),
                'edge_index': torch.from_numpy(data['edge_index']).long().to(device)
            },
            'ph_features': torch.from_numpy(data['ph_features'][None, ...]).float().to(device),
            'label': torch.tensor([labels[i]]).to(device)
        }
        
        # Forward
        optimizer.zero_grad()
        logits = model(batch)
        loss = F.cross_entropy(logits, batch['label'])
        
        # Backward
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(processed_data)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{n_epochs}: Loss = {avg_loss:.4f}")

print("\n✓ Training complete!")

## 6. Plot Training Loss

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, n_epochs+1), losses, marker='o', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss (Demo on 5 GIST Cases)')
plt.grid(alpha=0.3)
plt.show()

print(f"Initial loss: {losses[0]:.4f}")
print(f"Final loss: {losses[-1]:.4f}")
print(f"Reduction: {100*(losses[0]-losses[-1])/losses[0]:.1f}%")

## 7. Test Inference

In [ ]:
model.eval()

print("\nTest Predictions:")
print("=" * 50)

with torch.no_grad():
    for i, data in enumerate(processed_data):
        # Prepare batch
        batch = {
            'volume': torch.from_numpy(data['volume'][None, None, ...]).float().to(device),
            'mask': torch.from_numpy(data['mask'][None, None, ...]).float().to(device),
            'rim': torch.from_numpy(data['rim'][None, None, ...]).float().to(device),
            'mesh_data': {
                'vertices': torch.from_numpy(data['vertices']).float().to(device),
                'features': torch.from_numpy(data['node_features']).float().to(device),
                'edge_index': torch.from_numpy(data['edge_index']).long().to(device)
            },
            'ph_features': torch.from_numpy(data['ph_features'][None, ...]).float().to(device),
        }
        
        # Forward
        logits = model(batch)
        probs = torch.softmax(logits, dim=-1)
        pred_class = logits.argmax(dim=-1).item()
        confidence = probs[0, pred_class].item()
        
        print(f"{data['case_id']:15s} → Class {pred_class} (confidence: {confidence:.3f})")

print("=" * 50)

## 8. Extract Embeddings

In [ ]:
# Extract embeddings from all cases
all_embeddings = []

with torch.no_grad():
    for data in processed_data:
        batch = {
            'volume': torch.from_numpy(data['volume'][None, None, ...]).float().to(device),
            'mask': torch.from_numpy(data['mask'][None, None, ...]).float().to(device),
            'rim': torch.from_numpy(data['rim'][None, None, ...]).float().to(device),
            'mesh_data': {
                'vertices': torch.from_numpy(data['vertices']).float().to(device),
                'features': torch.from_numpy(data['node_features']).float().to(device),
                'edge_index': torch.from_numpy(data['edge_index']).long().to(device)
            },
            'ph_features': torch.from_numpy(data['ph_features'][None, ...]).float().to(device),
        }
        
        embeddings = model.get_embeddings(batch)
        all_embeddings.append(embeddings['fused'].cpu().numpy())

all_embeddings = np.vstack(all_embeddings)
print(f"\nExtracted embeddings: {all_embeddings.shape}")
print(f"These can be used for:")
print("  - UMAP/t-SNE visualization")
print("  - TabPFN classification")
print("  - Clustering analysis")

## 9. Visualize Embeddings (2D PCA)

In [ ]:
from sklearn.decomposition import PCA

# Reduce to 2D
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(all_embeddings)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    embeddings_2d[:, 0],
    embeddings_2d[:, 1],
    c=labels,
    cmap='tab10',
    s=200,
    alpha=0.7,
    edgecolors='black',
    linewidth=2
)

# Add labels
for i, data in enumerate(processed_data):
    plt.annotate(
        data['case_id'],
        (embeddings_2d[i, 0], embeddings_2d[i, 1]),
        fontsize=9,
        ha='right'
    )

plt.colorbar(scatter, label='Class')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('GeoTopo-STS Embeddings (2D PCA)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:
✅ Loading real GIST CT data  
✅ Preprocessing with geometry + topology  
✅ Creating GeoTopo-STS model  
✅ Training for 10 epochs  
✅ Making predictions  
✅ Extracting embeddings  
✅ Visualizing feature space  

---

## Next Steps for Real Training

1. **Preprocess all 246 cases**:
   ```bash
   python -m geotopo_sts.gist_data_loader --root ../data/gist --output ./preprocessed_gist --workers 4 --create-splits
   ```

2. **Add real labels** to `preprocessed_gist/labels.txt`

3. **Train full model**:
   ```bash
   python -m geotopo_sts.train --config config.yaml --data ./preprocessed_gist --output ./outputs/gist_full
   ```

4. **Evaluate**:
   ```bash
   python -m geotopo_sts.eval --config config.yaml --checkpoint ./outputs/gist_full/best_model.pth --data ./preprocessed_gist --ablations
   ```

**Your GeoTopo-STS pipeline is working!** 🚀